In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.crossover.ox import OrderCrossover
from pymoo.operators.mutation.inversion import InversionMutation
from pymoo.operators.sampling.rnd import PermutationRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.termination.collection import TerminationCollection
from pymoo.termination.default import DefaultSingleObjectiveTermination
from tqdm import tqdm

from qap_assignment.dataset import make_dataset, parse_dat_file
from qap_assignment.operators import SwapMutation
from qap_assignment.problem import QAP

make_dataset()

Prof let me use other termination conditions besides 50,000 gens

In [42]:
def run(seed):
    n, A, B = parse_dat_file("kra30a")
    problem = QAP(n, A, B)

    algorithm = GA(
        pop_size=300,
        sampling=PermutationRandomSampling(),
        # `Crossover` uses prob=0.9 by default anyway
        crossover=OrderCrossover(prob=0.9),
        # mutation=InversionMutation(prob=0.2),
        mutation=SwapMutation(
            prob=0.2
        ),  # 0.4 swap prob, 0.95 crossover is the best i've gotten so far
        # By default, tournament selection and elitist survival is used
    )

    termination = TerminationCollection(
        DefaultSingleObjectiveTermination(
            period=1500,  # let's try this
            n_max_gen=50000,
            # Make n_max_evals really big since I don't care about this
            n_max_evals=2**64,
        ),
        # Why is fmin not documented...
        # Shouldn't hardcode this, kra30a has a known optimum
        get_termination("fmin", 88900),
    )

    start_time = time.perf_counter()
    res = minimize(problem, algorithm, termination, seed=seed, verbose=False)
    end_time = time.perf_counter()
    return {
        "cost": res.F[0],
        "time": end_time - start_time,
    }

In [43]:
seed_sequence = np.random.SeedSequence(42)
run_seeds = seed_sequence.generate_state(50)
tasks = (delayed(run)(seed) for seed in run_seeds)
kra_results = []
for res in tqdm(
    Parallel(n_jobs=8, return_as="generator_unordered")(tasks),
    total=50,
):
    kra_results.append(res)

100%|██████████| 50/50 [16:31<00:00, 19.84s/it]  


In [ ]:
avg_cost = np.average([r["cost"] for r in kra_results])
avg_time = np.average([r["time"] for r in kra_results])
print(avg_cost)
print(avg_time)
print(np.min([r["cost"] for r in kra_results]))
# With OX crossover 0.9, swap mutation 0.2
# 91668.8
# 147.18628515798133
# minimum 89700.0
# With OX crossover 0.95, swap mutation 0.4
# 91736.2
# 129.90517527197954
# minimum 90160.0

91668.8
147.18628515798133
89700.0


In [35]:
def run_tai(seed):
    n, A, B = parse_dat_file("tai40a")
    problem = QAP(n, A, B)

    algorithm = GA(
        pop_size=300,
        sampling=PermutationRandomSampling(),
        # `Crossover` uses prob=0.9 by default anyway
        crossover=OrderCrossover(prob=0.95),
        # mutation=InversionMutation(prob=0.2),
        mutation=SwapMutation(
            prob=0.4
        ),  # 0.4 swap prob, 0.95 crossover is the best i've gotten so far
        # By default, tournament selection and elitist survival is used
    )

    termination = TerminationCollection(
        DefaultSingleObjectiveTermination(
            period=1500,  # let's try this
            n_max_gen=50000,
            # Make n_max_evals really big since I don't care about this
            n_max_evals=2**64,
        ),
    )

    start_time = time.perf_counter()
    res = minimize(problem, algorithm, termination, seed=seed, verbose=True)
    end_time = time.perf_counter()
    return {
        "cost": res.F[0],
        "time": end_time - start_time,
    }

In [36]:
seed_sequence = np.random.SeedSequence(42)
run_seeds = seed_sequence.generate_state(50)
tasks = (delayed(run_tai)(seed) for seed in run_seeds)
results = []
for res in tqdm(
    Parallel(n_jobs=8, return_as="generator_unordered")(tasks),
    total=50,
):
    results.append(res)

100%|██████████| 50/50 [14:17<00:00, 17.14s/it] 


In [37]:
avg_cost = np.average([r["cost"] for r in results])
avg_time = np.average([r["time"] for r in results])
print(avg_cost)
print(avg_time)
print(np.min([r["cost"] for r in results]))
# With OX crossover 0.9, swap mutation 0.2
# 3247817.68
# 140.9817344360007
# minimum 3212730
# With OX crossover 0.95, swap mutation 0.4
# 3234869.4
# 136.56625888799317
# minimum 3185812
# I get worse performance with 0.9 and 0.2, so use 0.95 and 0.4

3234869.4
126.02932803999167
3185812.0
